In [1]:
# Ermöglicht das erneute Laden der .py Module, auch nach deren Modifikation ohne den Kernel neu starten zu müssen

%load_ext autoreload
%autoreload 2


In [2]:
import json
import sqlite3
import requests
import re
from html import unescape
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
from urllib.parse import urlparse
from src.tagesschau_client import TagesschauClient
import os
import hashlib
import uuid



In [40]:
#from tagesschau_client import TagesschauClient

from dotenv import load_dotenv
load_dotenv()

from src.tagesschau_client import TagesschauClient

client = TagesschauClient(
    api_config_path="config/api_config.json",
    regions_path="config/regions.json",
    source_regions_path="config/source_regions.json",
    url_region_keywords_path="config/url_region_keywords.json",
    filters_path="config/filters.json",
)

await client.collect_and_store()



🕒 Ingest watermark (from ingest_date): 2026-01-17T18:21:35

📊 TAGESSCHAU INGEST SUMMARY
🔹 Artikel von API (Index): 102
🕒 Nach Watermark relevant: 74
💾 Artikel gespeichert:     74
📄 Kein Fulltext:           0
❌ Fehlgeschlagen:          0
⏭️ Gefiltert (Typ):         28
⏭️ Gefiltert (Ressort):     0
⏭️ Gefiltert (Watermark):   0


In [43]:
import os
import asyncio
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from pathlib import Path
import libsql_client

# ----------------------------
# Config
# ----------------------------
QDRANT_PATH = Path("vector_store/qdrant_turso_test4")
COLLECTION = "turso_articles"
LIMIT = 5

# ----------------------------
# Env
# ----------------------------
TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

# ----------------------------
# Embedding model
# ----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = 384

# ----------------------------
# Qdrant
# ----------------------------
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
qdrant = QdrantClient(path=str(QDRANT_PATH))

if COLLECTION not in [c.name for c in qdrant.get_collections().collections]:
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )

# ----------------------------
# Load 5 articles from Turso
# ----------------------------
async def main():
    print("🔌 Connecting to Turso...")
    db = libsql_client.create_client(
        url=TURSO_DB_URL,
        auth_token=TURSO_AUTH_TOKEN,
    )

    print("📥 Loading 5 articles from DB...")

    rs = await db.execute("""
    SELECT
        external_id,
        title,
        fulltext
    FROM articles
    WHERE
        title LIKE '%Auto%'
        OR title LIKE '%BMW%'
        OR title LIKE '%VW%'
        OR title LIKE '%Tesla%'
        OR title LIKE '%Wirtschaft%'
        OR title LIKE '%Industrie%'
    LIMIT 10
""")


    rows = rs.rows
    print(f"Loaded {len(rows)} articles.")

    # ----------------------------
    # Build embeddings
    # ----------------------------
    texts = []
    metadatas = []

    for r in rows:
        text = (r["title"] or "") + "\n\n" + (r["fulltext"] or "")
        texts.append(text)
        metadatas.append({
            "external_id": r["external_id"],
            "title": r["title"],
        })

    print("🧠 Computing embeddings...")
    vectors = model.encode(texts)

    # ----------------------------
    # Insert into Qdrant
    # ----------------------------
    points = []
    for i, (vec, meta) in enumerate(zip(vectors, metadatas)):
        points.append(
            PointStruct(
                id=i,
                vector=vec.tolist(),
                payload=meta,
            )
        )

    qdrant.upsert(collection_name=COLLECTION, points=points)

    print("✅ Inserted into Qdrant.")

    # ----------------------------
    # Query
    # ----------------------------
    query_text = "Was gibt es Neues zu Autoherstellern und Wirtschaft?"

    print("\n🔍 Query:", query_text)

    qvec = model.encode([query_text])[0]

    res = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec.tolist(),
        limit=5,
    )

    print("\n📄 Results:\n")

    for r in res.points:
        print(f"Score: {r.score:.4f}")
        print("external_id:", r.payload["external_id"])
        print("title:", r.payload["title"])
        print("-" * 60)

    await db.close()


if __name__ == "__main__":
    await main()


🔌 Connecting to Turso...
📥 Loading 5 articles from DB...
Loaded 10 articles.
🧠 Computing embeddings...
✅ Inserted into Qdrant.

🔍 Query: Was gibt es Neues zu Autoherstellern und Wirtschaft?

📄 Results:

Score: 0.3183
external_id: tagesschau_fm-story-swr-2c3825fd-4d8c-3e60-b189-39175f3808ca
title: Auto steht quer: Sperrung auf A81 nach Unfall bei Neuenstadt
------------------------------------------------------------
Score: 0.3122
external_id: c4f1fdef-8f2c-4cfe-86aa-7b2e1657f0e0
title: IHK zu Schwerin zieht negative Wirtschaftsbilanz für 2025
------------------------------------------------------------
Score: 0.3020
external_id: af069799-d332-434c-b892-220241325026
title: Landwirte wollen in MV erneut Autobahnauffahrten blockieren
------------------------------------------------------------
Score: 0.2985
external_id: tagesschau_fm-story-rbb_brandenburg-landwirte-proteste-autobahn-auffahrten-mercosur
title: Landwirte in Brandenburg kündigen Proteste an Autobahnen an
--------------------

In [50]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()

qdrant = QdrantClient(
    url=os.environ["QADRANT_ENDPOINT"],
    api_key=os.environ["QADRANT_API_KEY"]
)



In [53]:
print(qdrant.get_collections())


collections=[]


In [54]:
from sentence_transformers import SentenceTransformer

class LocalEmbedder:
    def __init__(self, model_name="BAAI/bge-m3"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        texts = [f"passage: {t}" for t in texts]
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text: str) -> list[float]:
        return self.model.encode([f"query: {text}"])[0].tolist()



In [47]:
import os
import uuid
import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Any

from dotenv import load_dotenv
import libsql_client
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from openai import OpenAI

# ----------------------------
# Env
# ----------------------------
load_dotenv()

TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

QDRANT_URL = os.environ["QADRANT_ENDPOINT"]
QDRANT_API_KEY = os.environ["QADRANT_API_KEY"]

OPENAI_API_KEY = os.environ["OPEN_AI_KEY"]

# ----------------------------
# Paths
# ----------------------------
SCHEMA_PATH = "config/article_schema.json"
PROMPTS_PATH = "config/prompts.json"

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ----------------------------
# Index / Kosten-Schutz Config
# ----------------------------
COLLECTION = "dummy_II"
EMBED_DIM = 1536

BATCH_SIZE = 100
MAX_ARTICLES = 2000
DRY_RUN = False

FILTER_KEYWORDS = ["Iran"]

MIN_DATE = "2026-01-23"
MAX_DATE = "2026-01-25"

TOP_K = 5  # retrieval count

# ----------------------------
# Helpers
# ----------------------------
def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_result_to_json(data: dict, prefix: str = "rag_result") -> Path:
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    path = OUTPUT_DIR / f"{prefix}_{ts}.json"

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"💾 Saved result to: {path}")
    return path


def safe_row_get(row: Any, key: str, default=None):
    """libsql_client Row behaves like a mapping via row[key], but has no .get()."""
    try:
        return row[key]
    except Exception:
        return default


def escape_like(s: str) -> str:
    # minimal escaping for SQL string literal; you are building SQL strings directly
    return s.replace("'", "''")


# ----------------------------
# OpenAI Clients
# ----------------------------
class OpenAIEmbedder:
    def __init__(self, api_key: str, model: str = "text-embedding-3-small"):
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        resp = self.client.embeddings.create(model=self.model, input=texts)
        return [d.embedding for d in resp.data]

    def embed_query(self, text: str) -> list[float]:
        resp = self.client.embeddings.create(model=self.model, input=[text])
        return resp.data[0].embedding


class OpenAISummarizer:
    def __init__(self, api_key: str, prompts: dict, prompt_key: str, model: str = "gpt-4.1-mini"):
        self.client = OpenAI(api_key=api_key)
        self.model = model

        if prompt_key not in prompts:
            raise ValueError(f"Prompt key '{prompt_key}' not found in prompts JSON")

        self.system_template = prompts[prompt_key]["system"]
        self.user_template = prompts[prompt_key]["user"]

    def summarize(self, query: str, documents: list[dict]) -> dict:
        """
        Returns a structured dict:
        {
          "summary": "...",
          "claims": [{"text": "...", "sources": [1,2]}],
          "raw": "<model output>"
        }
        If parsing fails, claims may be empty.
        """
        blocks = []
        for i, doc in enumerate(documents, 1):
            blocks.append(
                f"[{i}]\n"
                f"Titel: {doc.get('title','')}\n"
                f"Datum: {doc.get('published_at','')}\n"
                f"URL: {doc.get('url','')}\n"
                f"Inhalt:\n{doc.get('text','')}\n"
            )
        documents_text = "\n\n".join(blocks)

        system_prompt = self.system_template
        user_prompt = self.user_template.format(query=query, documents=documents_text)

        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.2,
        )

        raw = (resp.choices[0].message.content or "").strip()

        # Minimal, robust extraction: look for "ZUSAMMENFASSUNG:" and "AUSSAGEN:"
        summary_text = raw
        claims: list[dict] = []

        def _section(name: str) -> str:
            pat = rf"{name}\s*:\s*"
            m = __import__("re").search(pat, raw, flags=__import__("re").IGNORECASE)
            if not m:
                return ""
            start = m.end()
            # end at next ALLCAPS section label
            m2 = __import__("re").search(r"\n[A-ZÄÖÜ][A-ZÄÖÜ \-]{2,}:\s*\n", raw[start:], flags=__import__("re").MULTILINE)
            end = start + (m2.start() if m2 else len(raw[start:]))
            return raw[start:end].strip()

        sec_summary = _section("ZUSAMMENFASSUNG")
        sec_claims = _section("AUSSAGEN")

        if sec_summary:
            summary_text = sec_summary

        if sec_claims:
            # Parse lines like: "- Text ... [1,2]"
            import re

            for line in sec_claims.splitlines():
                line = line.strip()
                if not line:
                    continue
                if line.startswith("-"):
                    line = line[1:].strip()
                m = re.search(r"\[(.*?)\]\s*$", line)
                if m:
                    src_raw = m.group(1)
                    src = []
                    for part in src_raw.split(","):
                        part = part.strip()
                        if part.isdigit():
                            src.append(int(part))
                    text = re.sub(r"\s*\[(.*?)\]\s*$", "", line).strip()
                    if text:
                        claims.append({"text": text, "sources": src})
                else:
                    claims.append({"text": line, "sources": []})

        return {"summary": summary_text.strip(), "claims": claims, "raw": raw}


# ----------------------------
# Schema Handling
# ----------------------------
def get_columns_by_role(schema: dict, role: str) -> list[str]:
    return [col for col, spec in schema["columns"].items() if spec["role"] == role]


def get_select_columns(schema: dict) -> list[str]:
    return [col for col, spec in schema["columns"].items() if spec["role"] != "ignore"]


def build_select_sql(schema: dict, where_clause: str, limit: int, offset: int) -> str:
    cols = get_select_columns(schema)
    table = schema["table"]
    return f"""
        SELECT {", ".join(cols)}
        FROM {table}
        WHERE {where_clause}
        LIMIT {limit} OFFSET {offset}
    """


def build_embedding_text(row, schema: dict) -> str:
    parts = []
    for col in get_columns_by_role(schema, "embedding"):
        val = safe_row_get(row, col)
        if val:
            parts.append(str(val))
    return "\n\n".join(parts)


def build_payload(row, schema: dict) -> dict:
    payload = {}
    for col in get_columns_by_role(schema, "payload"):
        payload[col] = safe_row_get(row, col)
    # keep it clean: drop None values
    return {k: v for k, v in payload.items() if v is not None}


# ----------------------------
# SQL Filter
# ----------------------------
def build_where_clause() -> str:
    clauses = ["fulltext IS NOT NULL", "TRIM(fulltext) != ''"]

    if MIN_DATE:
        clauses.append(f"published_at >= '{escape_like(MIN_DATE)}'")
    if MAX_DATE:
        clauses.append(f"published_at <= '{escape_like(MAX_DATE)}'")

    if FILTER_KEYWORDS:
        like_parts = []
        for kw in FILTER_KEYWORDS:
            kw_esc = escape_like(kw)
            like_parts.append(f"title LIKE '%{kw_esc}%'")
            like_parts.append(f"fulltext LIKE '%{kw_esc}%'")
        clauses.append("(" + " OR ".join(like_parts) + ")")

    return " AND ".join(clauses)


# ----------------------------
# Load full texts for RAG
# ----------------------------
async def fetch_articles_by_ids(db, ids: list[str]) -> list[dict]:
    if not ids:
        return []

    # avoid SQL injection: we only accept ids from our own DB payloads
    ids_sql = ", ".join([f"'{escape_like(i)}'" for i in ids])

    sql = f"""
        SELECT external_id, title, published_at, url, fulltext
        FROM articles
        WHERE external_id IN ({ids_sql})
    """

    rs = await db.execute(sql)

    docs = []
    for r in rs.rows:
        docs.append(
            {
                "external_id": safe_row_get(r, "external_id"),
                "title": safe_row_get(r, "title"),
                "published_at": safe_row_get(r, "published_at"),
                "url": safe_row_get(r, "url"),
                "text": safe_row_get(r, "fulltext"),
            }
        )

    # preserve retrieval order as much as possible
    by_id = {d["external_id"]: d for d in docs}
    return [by_id[i] for i in ids if i in by_id]


# ----------------------------
# Main
# ----------------------------
async def main():
    schema = load_json(SCHEMA_PATH)
    prompts = load_json(PROMPTS_PATH)

    embedder = OpenAIEmbedder(OPENAI_API_KEY)
    summarizer = OpenAISummarizer(
        api_key=OPENAI_API_KEY,
        prompts=prompts,
        prompt_key="news_summary",
    )

    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    if not qdrant.collection_exists(COLLECTION):
        qdrant.create_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )

    print("🔌 Connecting to Turso...")
    db = libsql_client.create_client(url=TURSO_DB_URL, auth_token=TURSO_AUTH_TOKEN)

    where_clause = build_where_clause()
    print("🔍 WHERE:", where_clause)

    offset = 0
    total_indexed = 0

    # ----------------------------
    # Index
    # ----------------------------
    while True:
        if total_indexed >= MAX_ARTICLES:
            break

        sql = build_select_sql(schema, where_clause, BATCH_SIZE, offset)
        rs = await db.execute(sql)
        rows = rs.rows

        if not rows:
            break

        texts, payloads, ids = [], [], []

        for r in rows:
            if total_indexed >= MAX_ARTICLES:
                break

            text = build_embedding_text(r, schema)
            if not text.strip():
                continue

            payload = build_payload(r, schema)
            if "external_id" not in payload:
                continue

            texts.append(text)
            payloads.append(payload)

            pid = str(uuid.uuid5(uuid.NAMESPACE_URL, str(payload["external_id"])))
            ids.append(pid)

            total_indexed += 1

        if not texts:
            offset += BATCH_SIZE
            continue

        if DRY_RUN:
            print(f"🧪 DRY_RUN=True → skipping embeddings/upsert for {len(texts)} docs")
        else:
            vectors = embedder.embed_documents(texts)
            points = [
                PointStruct(id=ids[i], vector=vectors[i], payload=payloads[i])
                for i in range(len(ids))
            ]
            qdrant.upsert(collection_name=COLLECTION, points=points)

        offset += BATCH_SIZE

    # ----------------------------
    # Retrieval
    # ----------------------------
    keywords_str = " ".join(FILTER_KEYWORDS)
    query_text = f"Neuigkeiten zu {keywords_str}"

    qvec = embedder.embed_query(query_text)

    res = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec,
        limit=TOP_K,
    )

    retrieved = res.points
    external_ids = [p.payload["external_id"] for p in retrieved if p.payload and p.payload.get("external_id")]

    docs = await fetch_articles_by_ids(db, external_ids)

    # ----------------------------
    # Summary (structured + raw)
    # ----------------------------
    summary_pack = summarizer.summarize(query_text, docs)
    summary_text = summary_pack["summary"]
    claims = summary_pack["claims"]
    llm_raw = summary_pack["raw"]

    print(f"🧾 Summary length: {len(summary_text)} chars")

    # ----------------------------
    # JSON Result
    # ----------------------------
    result = {
        "query": query_text,
        "filters": {
            "keywords": FILTER_KEYWORDS,
            "min_date": MIN_DATE,
            "max_date": MAX_DATE,
        },
        "retrieved_documents": [
            {
                "rank": i,
                "score": float(p.score),
                "external_id": p.payload.get("external_id"),
                "title": p.payload.get("title"),
                "published_at": p.payload.get("published_at"),
                "url": p.payload.get("url"),
            }
            for i, p in enumerate(retrieved, 1)
        ],
        "sources": [
            {
                "index": i,
                "external_id": d["external_id"],
                "title": d["title"],
                "published_at": d["published_at"],
                "url": d["url"],
            }
            for i, d in enumerate(docs, 1)
        ],
        # ✅ summary is stored ONCE (no confusion)
        "summary": {
            "text": summary_text,
            "claims": claims,
            "raw_output": llm_raw,
        },
        "meta": {
            "collection": COLLECTION,
            "model_embedding": "text-embedding-3-small",
            "model_summary": "gpt-4.1-mini",
            "created_at": datetime.now(timezone.utc).isoformat(),
            "dry_run": DRY_RUN,
            "indexed_docs": total_indexed,
            "top_k": TOP_K,
        },
    }

    out_path = save_result_to_json(result, prefix="news_rag")

    print("\n🧠 ZUSAMMENFASSUNG:\n")
    print(summary_text)

    if claims:
        print("\n🧩 AUSSAGEN:\n")
        for c in claims:
            print(f"- {c['text']} {c.get('sources', [])}")

    print("\n📌 JSON CHECK:")
    print("file:", out_path)
    print("summary_in_json:", bool(result["summary"]["text"]))

    await db.close()


# ----------------------------
# Notebook Entrypoint
# ----------------------------
await main()


🔌 Connecting to Turso...
🔍 WHERE: fulltext IS NOT NULL AND TRIM(fulltext) != '' AND published_at >= '2026-01-23' AND published_at <= '2026-01-25' AND (title LIKE '%Iran%' OR fulltext LIKE '%Iran%')
🧾 Summary length: 1379 chars
💾 Saved result to: output/news_rag_20260125_140928.json

🧠 ZUSAMMENFASSUNG:

Im Iran haben seit Ende Dezember 2025 landesweite Proteste gegen die autoritäre Regierung stattgefunden, die sich aus wirtschaftlichen Problemen zu politischen Aufständen entwickelten. Die Sicherheitskräfte reagierten mit brutaler Gewalt, wobei nach Angaben von Menschenrechtsorganisationen und Aktivisten mehrere Tausend Menschen getötet wurden, darunter viele Demonstranten, Minderjährige und Unbeteiligte [1,3,5]. Die iranische Justiz kündigte Schnellverfahren und mögliche Hinrichtungen von Festgenommenen an, was international Besorgnis auslöst und von US-Präsident Trump mit Drohungen gegen das Regime beantwortet wurde [1,3,5]. Trotz der derzeitigen Ruhe auf den Straßen bleibt die Wut in 

In [36]:
import subprocess
import sys
import os
import re
import time

# 🔥 Kill any old streamlit processes (important!)
os.system("pkill -f streamlit")

# ⚙️ Path to your main.py
APP_PATH = "streamlit/main.py"   # ⬅️ ggf. anpassen!

cmd = [
    sys.executable, "-m", "streamlit", "run", APP_PATH,
    "--server.address", "0.0.0.0",
    "--server.port", "8501",
    "--server.headless", "true",
]

print("🚀 Starte Streamlit...")

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

url = None

start = time.time()
timeout = 30  # Sekunden

while time.time() - start < timeout:
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.1)
        continue

    print(line.rstrip())

    m = re.search(r"(Local URL|Network URL):\s+(http://\S+)", line)
    if m:
        url = m.group(2)
        break

if url:
    print("\n✅ Streamlit läuft hier:")
    print(url)
else:
    print("\n❌ Keine URL gefunden. Streamlit-Output oben prüfen.")


🚀 Starte Streamlit...



  You can now view your Streamlit app in your browser.

  URL: http://0.0.0.0:8501

2026-02-01 17:52:29.832 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-02-01 17:52:35.869 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-02-01 17:52:37.739 Please replace `use_container_width` with `width`.

❌ Keine URL gefunden. Streamlit-Output oben prüfen.


In [35]:
import os
os.system("pkill -f streamlit")


15

# Topic Clustering

In [8]:
import os
import time
import asyncio
import uuid
import json
from datetime import datetime
from pathlib import Path

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
import libsql_client
from dotenv import load_dotenv

# ------------------------------------------------------------
# ENV
# ------------------------------------------------------------
load_dotenv()

TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

QDRANT_URL = os.environ["QADRANT_ENDPOINT"]
QDRANT_API_KEY = os.environ["QADRANT_API_KEY"]

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
COLLECTION_NAME = "articles_new"
BATCH_SIZE = 500
MAX_CHARS = 2000
SLEEP_SEC = 0.2
VECTOR_DIM = 768
EMBED_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

SCHEMA_PATH = Path("config/article_schema.json")

# ------------------------------------------------------------
# LOAD ARTICLE SCHEMA
# ------------------------------------------------------------
with SCHEMA_PATH.open("r", encoding="utf-8") as f:
    ARTICLE_SCHEMA = json.load(f)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def stable_point_id(external_id: str) -> str:
    return str(uuid.uuid5(uuid.NAMESPACE_URL, external_id))


def build_payload(row: dict, schema: dict) -> dict:
    payload = {}
    for col, cfg in schema["columns"].items():
        if cfg["role"] == "payload":
            val = row.get(col)
            if val is not None:
                payload[col] = val
    return payload


def build_embedding_text(row: dict, schema: dict) -> str:
    parts = []
    for col, cfg in schema["columns"].items():
        if cfg["role"] == "embedding":
            val = row.get(col)
            if val:
                parts.append(str(val))
    return "\n\n".join(parts)

# ------------------------------------------------------------
# INIT
# ------------------------------------------------------------
embedder = SentenceTransformer(EMBED_MODEL_NAME)

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

db = libsql_client.create_client(
    url=TURSO_DB_URL,
    auth_token=TURSO_AUTH_TOKEN,
)

# ------------------------------------------------------------
# SQL
# ------------------------------------------------------------
SELECT_SQL = f"""
    SELECT *
    FROM articles
    WHERE fulltext IS NOT NULL
    ORDER BY published_at ASC
    LIMIT {BATCH_SIZE}
"""

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
async def main():

    # Ensure column exists
    try:
        await db.execute("""
            ALTER TABLE articles
            ADD COLUMN embedded_at TIMESTAMP
        """)
        print("➕ embedded_at column created")
    except Exception:
        print("✅ Schema geprüft (embedded_at existiert)")

    # ❗ KEIN collection_exists / create_collection
    print(f"ℹ️ Using existing Qdrant collection: {COLLECTION_NAME}")

    # Backfill
    while True:
        rs = await db.execute(SELECT_SQL)
        if not rs.rows:
            print("🎉 Alle Artikel sind embedded.")
            break

        # libsql_client → echtes dict
        columns = rs.columns
        rows = [dict(zip(columns, r)) for r in rs.rows]

        texts = [
            build_embedding_text(row, ARTICLE_SCHEMA)[:MAX_CHARS]
            for row in rows
        ]

        embeddings = embedder.encode(
            texts,
            batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        points = []
        external_ids = []

        for i, row in enumerate(rows):
            external_id = row["external_id"]
            external_ids.append(external_id)

            payload = build_payload(row, ARTICLE_SCHEMA)
            payload.update({
                "embedded_at": datetime.utcnow().isoformat(),
                "embedding_model": EMBED_MODEL_NAME
            })

            points.append(
                PointStruct(
                    id=stable_point_id(external_id),
                    vector=embeddings[i].tolist(),
                    payload=payload
                )
            )

        qdrant.upsert(
            collection_name=COLLECTION_NAME,
            points=points
        )

        placeholders = ",".join(["?"] * len(external_ids))
        await db.execute(
            f"""
            UPDATE articles
            SET embedded_at = CURRENT_TIMESTAMP
            WHERE external_id IN ({placeholders})
            """,
            external_ids
        )

        print(f"✔ embedded + stored + marked: {len(external_ids)}")
        time.sleep(SLEEP_SEC)

    await db.close()

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------
await main()


ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0xfffebf5d9250>


✅ Schema geprüft (embedded_at existiert)
ℹ️ Using existing Qdrant collection: articles_new
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
✔ embedded + stored + marked: 500
